# HP Quiz — RAG testing
Assumes the index has been built with `python build_index.py`.  
`all-MiniLM-L6-v2` embeddings · ChromaDB · TinyLlama-1.1B (local)

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline

CHAT_MODEL      = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "hp_books"
EMBED_MODEL     = "all-MiniLM-L6-v2"

embedder  = SentenceTransformer(EMBED_MODEL)
generator = pipeline("text-generation", model=CHAT_MODEL, dtype="auto", device_map="auto")

collection = chromadb.PersistentClient(path=CHROMA_PATH).get_collection(COLLECTION_NAME)
print(f"Index loaded — {collection.count():,} vectors")

In [ ]:
def retrieve(query: str, k: int = 5) -> list[str]:
    q_emb = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=k)
    return results["documents"][0]

In [ ]:
SYSTEM_PROMPT = "You are a Harry Potter expert. Answer concisely based only on the provided context."

def rag_query(question: str, k: int = 5) -> str:
    context = "\n\n---\n\n".join(retrieve(question, k))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ]
    prompt = generator.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    out = generator(prompt, max_new_tokens=256)
    return out[0]["generated_text"][len(prompt):].strip()

In [ ]:
print(rag_query("What is the address of the Dursleys?"))

In [ ]:
print(rag_query("Generate 3 multiple-choice quiz questions about Hogwarts."))